# Pull and inspect Alpaca market data

This notebook downloads 15-minute SPY and QQQ bars from Alpaca's IEX feed, creates a few features, and saves them for notebook 02. Run the cells from top to bottom. No trading client is used.

**One-time setup:** Install Python 3.10+ and run `python -m pip install alpaca-py pandas numpy plotly pyarrow python-dotenv ipykernel`. Put your Alpaca market-data keys in a `.env` file beside this notebook:

```text
API_KEY=your_key
SECRET_KEY=your_secret
```

You can also set those two environment variables in your operating system. Open Jupyter from the repository root or its `notebooks` folder. Each person needs their own Alpaca credentials and an internet connection. Missing credentials or a failed request stop the notebook with an error.


In [1]:
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from alpaca.data.enums import DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from dotenv import load_dotenv
from IPython.display import display

# Jupyter normally starts in either the repository root or its notebooks folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
load_dotenv(ROOT / "notebooks" / ".env")
load_dotenv(ROOT / ".env")

SYMBOLS = ["SPY", "QQQ"]
BAR_MINUTES = 15
END_UTC = datetime.now(timezone.utc)
START_UTC = END_UTC - timedelta(days=45)

key = os.getenv("API_KEY")
secret = os.getenv("SECRET_KEY")
if not key or not secret:
    raise ValueError("Set API_KEY and SECRET_KEY in notebooks/.env or your environment")


## 1. Download bars

The request uses Alpaca's historical stock-data client. Timestamps stay in UTC; we convert them to New York time only when calculating session features.


In [2]:
client = StockHistoricalDataClient(key, secret)
request = StockBarsRequest(
    symbol_or_symbols=SYMBOLS,
    timeframe=TimeFrame(BAR_MINUTES, TimeFrameUnit.Minute),
    start=START_UTC,
    end=END_UTC,
    feed=DataFeed.IEX,
)
bars = client.get_stock_bars(request).df.reset_index()
if bars.empty:
    raise ValueError("Alpaca returned no bars for this request")

bars["timestamp"] = pd.to_datetime(bars["timestamp"], utc=True)
bars = bars.sort_values(["symbol", "timestamp"]).drop_duplicates(["symbol", "timestamp"]).reset_index(drop=True)
display(bars.groupby("symbol").agg(rows=("timestamp", "size"), first=("timestamp", "min"), last=("timestamp", "max")))
display(bars.head())


,rows,first,last
symbol,,,
QQQ,1001,2026-08-03 12:00:00+00:00,2026-09-15 14:45:00+00:00
SPY,887,2026-08-03 12:00:00+00:00,2026-09-15 14:45:00+00:00


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,QQQ,2026-08-03 12:00:00+00:00,688.630,688.63,688.630,688.63,40.0,1.0,688.630000
1,QQQ,2026-08-03 12:30:00+00:00,689.020,689.02,689.020,689.02,40.0,1.0,689.020000
2,QQQ,2026-08-03 13:00:00+00:00,689.350,689.65,689.340,689.65,520.0,6.0,689.367968
3,QQQ,2026-08-03 13:30:00+00:00,688.290,691.25,686.025,690.20,33585.0,550.0,688.856056
4,QQQ,2026-08-03 13:45:00+00:00,690.335,693.19,689.260,693.19,47227.0,517.0,690.438024


## 2. Build features from current and past bars

VWAP is the volume-weighted average price so far in a session. The return and volatility columns use only bars already observed. Notebook 02 calculates future returns separately.


In [3]:
market_features = bars.copy()
ny_time = market_features["timestamp"].dt.tz_convert("America/New_York")
market_features["session_date"] = ny_time.dt.strftime("%Y-%m-%d")
market_features["time_of_day"] = ny_time.dt.strftime("%H:%M")
market_features["minute_of_day"] = ny_time.dt.hour * 60 + ny_time.dt.minute

by_symbol = market_features.groupby("symbol", sort=False)
by_session = market_features.groupby(["symbol", "session_date"], sort=False)
market_features["return_1bar"] = by_symbol["close"].pct_change(fill_method=None)
log_return = np.log(market_features["close"] / by_symbol["close"].shift(1))
market_features["log_return_1bar"] = log_return
market_features["realized_volatility_12bar"] = log_return.groupby(market_features["symbol"]).transform(
    lambda x: x.rolling(12, min_periods=6).std()
)

typical_price = (market_features["high"] + market_features["low"] + market_features["close"]) / 3
session_keys = [market_features["symbol"], market_features["session_date"]]
cum_dollars = (typical_price * market_features["volume"]).groupby(session_keys).cumsum()
cum_volume = by_session["volume"].cumsum().replace(0, np.nan)
market_features["intraday_vwap"] = cum_dollars / cum_volume
market_features["distance_from_vwap_pct"] = market_features["close"] / market_features["intraday_vwap"] - 1

new_session = market_features["session_date"].ne(by_symbol["session_date"].shift(1))
previous_close = by_symbol["close"].shift(1).where(new_session).groupby(market_features["symbol"]).ffill()
market_features["overnight_gap_pct"] = by_session["open"].transform("first") / previous_close - 1
market_features["volume_vs_prior_time_median"] = market_features["volume"] / (
    market_features.groupby(["symbol", "time_of_day"])["volume"]
    .transform(lambda x: x.shift(1).expanding().median())
)

display(market_features.head())


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,session_date,time_of_day,minute_of_day,return_1bar,log_return_1bar,realized_volatility_12bar,intraday_vwap,distance_from_vwap_pct,overnight_gap_pct,volume_vs_prior_time_median
0,QQQ,2026-08-03 12:00:00+00:00,688.630,688.63,688.630,688.63,40.0,1.0,688.630000,2026-08-03,08:00,480,NaN,NaN,NaN,688.630000,0.000000,NaN,NaN
1,QQQ,2026-08-03 12:30:00+00:00,689.020,689.02,689.020,689.02,40.0,1.0,689.020000,2026-08-03,08:30,510,0.000566,0.000566,NaN,688.825000,0.000283,NaN,NaN
2,QQQ,2026-08-03 13:00:00+00:00,689.350,689.65,689.340,689.65,520.0,6.0,689.367968,2026-08-03,09:00,540,0.000914,0.000914,NaN,689.450444,0.000289,NaN,NaN
3,QQQ,2026-08-03 13:30:00+00:00,688.290,691.25,686.025,690.20,33585.0,550.0,688.856056,2026-08-03,09:30,570,0.000798,0.000797,NaN,689.163460,0.001504,NaN,NaN
4,QQQ,2026-08-03 13:45:00+00:00,690.335,693.19,689.260,693.19,47227.0,517.0,690.438024,2026-08-03,09:45,585,0.004332,0.004323,NaN,690.739322,0.003548,NaN,NaN


## 3. Check the sample and save it

The chart shows whether trading activity changes during the day. The Parquet file is notebook 02's input; the CSV is convenient for inspection.


In [4]:
volume_by_time = market_features.groupby("time_of_day", as_index=False)["volume"].median()
fig = px.bar(volume_by_time, x="time_of_day", y="volume", title="Median 15-minute volume by New York time")
fig.update_layout(xaxis_title="New York time", yaxis_title="Median shares")
fig.show()

output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
parquet_path = output_dir / "market_features.parquet"
csv_path = output_dir / "market_features.csv"
market_features.to_parquet(parquet_path, index=False)
market_features.to_csv(csv_path, index=False)
print(f"Saved {len(market_features):,} rows to {parquet_path}")
print(f"CSV copy: {csv_path}")


Saved 1,888 rows to C:\Users\maxme\OneDrive\Documents\Mizzou\Orgs\TigerQuant\TigerQuant-Slides-Pipeline\data\processed\market_features.parquet
CSV copy: C:\Users\maxme\OneDrive\Documents\Mizzou\Orgs\TigerQuant\TigerQuant-Slides-Pipeline\data\processed\market_features.csv


## 4. View recent SPY candles

Each candle shows one 15-minute bar: the body runs from open to close, and the wick marks the high and low. The chart uses the Alpaca bars already downloaded above.

In [5]:
import plotly.graph_objects as go

spy = bars.loc[bars["symbol"].eq("SPY")].copy()
spy["session_date"] = spy["timestamp"].dt.tz_convert("America/New_York").dt.date
recent_sessions = sorted(spy["session_date"].unique())[-5:]
recent_spy = spy.loc[spy["session_date"].isin(recent_sessions)]

fig = go.Figure(go.Candlestick(
    x=recent_spy["timestamp"].dt.tz_convert("America/New_York"),
    open=recent_spy["open"],
    high=recent_spy["high"],
    low=recent_spy["low"],
    close=recent_spy["close"],
    name="SPY",
))
fig.update_layout(
    title="SPY — last five trading sessions (15-minute IEX bars)",
    xaxis_title="New York time",
    yaxis_title="Price (USD)",
    xaxis_rangeslider_visible=False,
    height=550,
)
fig.show()